In [1]:
# 1. 导入依赖并加载电路类
import sympy as sp
from IPython.display import display
from build_circuit_graph_rebuild import Circuit

sp.init_printing(use_latex='mathjax')

In [2]:
# 2. 创建空电路并逐步添加元件
circuit = Circuit()

jj1 = circuit.add_component(0, 2, 'JJ', 'EJ1')
jj2 = circuit.add_component(1, 2, 'JJ', 'EJ2')
L = circuit.add_component(0, 1, 'L', 'L')
C1 = circuit.add_component(0, 2, 'C', 'C1')
C2 = circuit.add_component(1, 2, 'C', 'C2')

In [3]:
# 2.1 互感示例
circuit_mutual = Circuit()

L1_m = circuit_mutual.add_component(0, 1, 'L', 'L1_m')
L2_m = circuit_mutual.add_component(1, 2, 'L', 'L2_m')
C_m = circuit_mutual.add_component(0, 2, 'C', 'C_m')

mid_m = circuit_mutual.add_mutual(L1_m, L2_m, 'M12_m')

print(f'已添加互感: mutual_id={mid_m}, 关联电感元件ID=({C_m}, {L2_m})')
circuit_mutual.print_edges()

已添加互感: mutual_id=0, 关联电感元件ID=(2, 1)
树枝数: 2, 连支数: 1, 总支路数: 3
  Key 0 [树枝]: L (0->1) 值: L1_m [元件ID: 0]
  Key 1 [树枝]: L (1->2) 值: L2_m [元件ID: 1]
  Key 2 [连支]: C (0->2) 值: C_m [元件ID: 2]


In [4]:
# 3. 打印元件列表
circuit.print_components()

共 5 个元件:
  元件 0: JJ (0-2) 值: EJ1
  元件 1: JJ (1-2) 值: EJ2
  元件 2: L (0-1) 值: L
  元件 3: C (0-2) 值: C1
  元件 4: C (1-2) 值: C2


In [5]:
# 4. 打印重排后的边信息
circuit.print_edges()

树枝数: 2, 连支数: 3, 总支路数: 5
  Key 0 [树枝]: JJ (0->2) 值: EJ1 [元件ID: 0]
  Key 1 [树枝]: JJ (1->2) 值: EJ2 [元件ID: 1]
  Key 2 [连支]: L (0->1) 值: L [元件ID: 2]
  Key 3 [连支]: C (0->2) 值: C1 [元件ID: 3]
  Key 4 [连支]: C (1->2) 值: C2 [元件ID: 4]


In [6]:
# 5. 打印基本回路（尚未配置物理回路, 外磁通显示 "未确定"）
circuit.print_loops()

电路共有 3 个基本回路 (树枝数: 2, 总支路数: 5)

回路 0: (由连支 Key 2 闭合)
  连支: L (0->1), 值: L [元件ID: 2], 遍历: 0→1 (+1)
  回路路径: 0 -> 1 -> 2 -> 0
  包含树枝:
    Key 1: JJ (1->2), 值: EJ2 [元件ID: 1], 遍历: 1→2 (+1)
    Key 0: JJ (0->2), 值: EJ1 [元件ID: 0], 遍历: 2→0 (-1)
  外磁通: 未确定

回路 1: (由连支 Key 3 闭合)
  连支: C (0->2), 值: C1 [元件ID: 3], 遍历: 0→2 (+1)
  回路路径: 0 -> 2 -> 0
  包含树枝:
    Key 0: JJ (0->2), 值: EJ1 [元件ID: 0], 遍历: 2→0 (-1)
  外磁通: 未确定

回路 2: (由连支 Key 4 闭合)
  连支: C (1->2), 值: C2 [元件ID: 4], 遍历: 1→2 (+1)
  回路路径: 1 -> 2 -> 1
  包含树枝:
    Key 1: JJ (1->2), 值: EJ2 [元件ID: 1], 遍历: 2→1 (-1)
  外磁通: 未确定

[注意] 需要恰好 3 个线性无关的物理回路来确定外磁通，当前已提供 0 个。即使磁通为 0 的回路也必须添加 (使用 add_physical_flux(..., 0))。


In [17]:
circuit.remove_physical_flux(3)

In [20]:
# 6. 添加所有物理回路
#    本电路有 3 个基本回路 (连支数=3), 必须提供恰好 3 个线性无关的物理回路
#    即使磁通为 0 也要添加, 以便唯一解出各基本回路的外磁通

print(f"需要提供 {circuit.num_loops} 个物理回路\n")

# 物理回路 1: JJ1-JJ2-L 围成的三角形网孔 (0→2→1→0), 磁通 Phi_ext
circuit.add_physical_flux([C1, C2, L], 'Phi_ext')

# 物理回路 2: JJ1 // C1 并联 (节点 0-2), 磁通 = 0
circuit.add_physical_flux([jj1, C1], 'a', direction=1)

# 物理回路 3: JJ2 // C2 并联 (节点 1-2), 磁通 = 0
circuit.add_physical_flux([jj2, C2], 0, direction=1)

circuit.print_physical_fluxes()

需要提供 3 个物理回路

共 3 个物理磁通:
  磁通 1: Φ = a
    回路元件: [0 -> 3] = JJ(0→2) -> C(2→0)
  磁通 2: Φ = 0
    回路元件: [1 -> 4] = JJ(1→2) -> C(2→1)
  磁通 4: Φ = 3*Phi_ext
    回路元件: [2 -> 3 -> 4] = L(1→0) -> C(0→2) -> C(2→1)


In [21]:
# 7. 打印基本回路（含线性求解得到的外磁通）
#    矩阵方程 A · φ_ext = b 被自动求解
circuit.print_loops()

电路共有 3 个基本回路 (树枝数: 2, 总支路数: 5)

回路 0: (由连支 Key 2 闭合)
  连支: L (0->1), 值: L [元件ID: 2], 遍历: 0→1 (+1)
  回路路径: 0 -> 1 -> 2 -> 0
  包含树枝:
    Key 1: JJ (1->2), 值: EJ2 [元件ID: 1], 遍历: 1→2 (+1)
    Key 0: JJ (0->2), 值: EJ1 [元件ID: 0], 遍历: 2→0 (-1)
  外磁通: -3*Phi_ext - a
  磁通来源: -1 × 3*Phi_ext (磁通ID 4)

回路 1: (由连支 Key 3 闭合)
  连支: C (0->2), 值: C1 [元件ID: 3], 遍历: 0→2 (+1)
  回路路径: 0 -> 2 -> 0
  包含树枝:
    Key 0: JJ (0->2), 值: EJ1 [元件ID: 0], 遍历: 2→0 (-1)
  外磁通: -a
  磁通来源: -1 × a (磁通ID 1), +1 × 3*Phi_ext (磁通ID 4)

回路 2: (由连支 Key 4 闭合)
  连支: C (1->2), 值: C2 [元件ID: 4], 遍历: 1→2 (+1)
  回路路径: 1 -> 2 -> 1
  包含树枝:
    Key 1: JJ (1->2), 值: EJ2 [元件ID: 1], 遍历: 2→1 (-1)
  外磁通: 0
  磁通来源: -1 × 0 (磁通ID 2), -1 × 3*Phi_ext (磁通ID 4)



In [9]:
# 8. 显示基本割集矩阵 F_C
display(circuit.F_C)

⎡1  0  1   1  0⎤
⎢              ⎥
⎣0  1  -1  0  1⎦

In [10]:
# 9. 计算并展示哈密顿量及关键中间量
H, info = circuit.hamiltonian()

print('广义坐标 (树枝磁通):')
display(info['phi_t'])

print('外磁通变量:')
if info['ext_fluxes']:
    display(sp.Matrix(info['ext_fluxes']))
else:
    print('无')

print('全支路磁通 Phi_vec:')
display(info['Phi_vec'])

print('哈密顿量 H:')
display(sp.expand(H))

广义坐标 (树枝磁通):


⎡Φₜ ₀⎤
⎢    ⎥
⎣Φₜ ₁⎦

外磁通变量:


[Φₑₓₜ]

全支路磁通 Phi_vec:


⎡       Φₜ ₀        ⎤
⎢                   ⎥
⎢       Φₜ ₁        ⎥
⎢                   ⎥
⎢-Φₑₓₜ + Φₜ ₀ - Φₜ ₁⎥
⎢                   ⎥
⎢       Φₜ ₀        ⎥
⎢                   ⎥
⎣       Φₜ ₁        ⎦

哈密顿量 H:


                                     2                               2         ↪
                                 Φₑₓₜ    Φₑₓₜ⋅Φₜ ₀   Φₑₓₜ⋅Φₜ ₁   Φₜ ₀    Φₜ ₀⋅ ↪
-EJ₁⋅cos(Φₜ ₀) - EJ₂⋅cos(Φₜ ₁) + ───── - ───────── + ───────── + ───── - ───── ↪
                                  2⋅L        L           L        2⋅L        L ↪

↪            2       2       2
↪ Φₜ ₁   Φₜ ₁    Qₜ ₁    Qₜ ₀ 
↪ ──── + ───── + ───── + ─────
↪         2⋅L    2⋅C₂    2⋅C₁ 